# Week 6 – Advanced Python Analysis: AAPL Historical Stock Data
**AnalystLab Africa Internship | Batch B | Chidozie**

**Objective:** Apply advanced Pandas techniques, time-series analysis, and feature engineering
to 5 years of Apple Inc. (AAPL) historical stock market data downloaded from Yahoo Finance.

**Dataset:** AAPL Historical Data (5-year range) — Date, Open, High, Low, Close, Adj Close, Volume


## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Libraries loaded successfully.")


## 1. Data Loading

> **Before running:** place the CSV file you downloaded from Yahoo Finance
> (5-year range, `Historical Data > Download`) in the same folder as this notebook
> and update `FILE_PATH` below. A small sample file (`AAPL_sample.csv`) is included
> so you can test the notebook end-to-end before swapping in your real download —
> **replace it with your actual Yahoo Finance file before submitting.**


In [ ]:
FILE_PATH = "AAPL_sample.csv"   # <-- replace with your downloaded file, e.g. "AAPL.csv"

df = pd.read_csv(FILE_PATH)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()


### 1b. Alternative: Pull data directly in Colab with `yfinance` (no manual download needed)

If you'd rather not download a CSV from Yahoo Finance manually (e.g. Gold-subscription CSV download is blocked), run the cell below in Colab instead. It installs and uses the `yfinance` package to pull the same Yahoo Finance data directly into a DataFrame with 5 years of daily history, and saves it as `AAPL.csv` so the rest of the notebook (which reads from `FILE_PATH`) works unchanged.

**Use only ONE of the two loading methods** — either the CSV upload above, or this cell.

In [ ]:
# Run this cell in Colab to fetch AAPL data directly from Yahoo Finance (no manual CSV download)
!pip install -q yfinance

import yfinance as yf

data = yf.download("AAPL", period="5y", interval="1d", auto_adjust=False)
data = data.reset_index()

# yfinance sometimes returns MultiIndex columns when auto_adjust=False with a single ticker; flatten if so
if isinstance(data.columns, pd.MultiIndex):
    data.columns = [c[0] if c[0] else c[1] for c in data.columns]

data.to_csv("AAPL.csv", index=False)
print("Saved AAPL.csv with", len(data), "rows")
data.head()

# After this runs, set FILE_PATH = "AAPL.csv" in the cell above (Section 1) and re-run from there.

## 2. Data Exploration

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Check missing values and duplicate rows
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDate range:", df['Date'].min(), "to", df['Date'].max())


## 3. Data Cleaning & Preprocessing

In [ ]:
# Convert Date to datetime and sort chronologically
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Ensure numeric columns are numeric (Yahoo exports sometimes include 'Dividend'/'Split' rows or commas)
num_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', ''), errors='coerce')

# Drop rows that are entirely missing on price data, then forward-fill any small gaps
df = df.dropna(subset=['Close'])
df[num_cols] = df[num_cols].ffill()

# Remove exact duplicate rows
df = df.drop_duplicates(subset='Date')

df = df.set_index('Date')
print("Cleaned shape:", df.shape)
df.head()


## 4. Data Transformation with Pandas
Filtering, sorting, grouping/aggregation, calculated columns, and restructuring.

In [ ]:
# Calculated column: daily trading range
df['Daily_Range'] = df['High'] - df['Low']

# Filtering: most recent 1 year of data
last_year = df[df.index >= df.index.max() - pd.DateOffset(years=1)]
print("Rows in the most recent year:", len(last_year))

# Sorting: 10 highest-volume trading days
top_volume_days = df.sort_values('Volume', ascending=False).head(10)
top_volume_days[['Close', 'Volume']]


In [ ]:
# Grouping & aggregation: yearly summary statistics
yearly_summary = df.groupby(df.index.year).agg(
    Avg_Close=('Close', 'mean'),
    Max_Close=('Close', 'max'),
    Min_Close=('Close', 'min'),
    Avg_Volume=('Volume', 'mean'),
    Trading_Days=('Close', 'count')
).round(2)
yearly_summary


In [ ]:
# Restructuring: resample to monthly OHLC (business-standard aggregation)
monthly_ohlc = df['Close'].resample('ME').agg(['first', 'max', 'min', 'last'])
monthly_ohlc.columns = ['Month_Open', 'Month_High', 'Month_Low', 'Month_Close']
monthly_ohlc.tail(12)


## 5. Time-Series Analysis
Date manipulation, trend analysis, rolling averages, and percentage change.

In [ ]:
# Date/time manipulation
df['Year'] = df.index.year
df['Month'] = df.index.month
df['Weekday'] = df.index.day_name()

# Daily percentage change
df['Daily_Return_%'] = df['Close'].pct_change() * 100

df[['Close', 'Year', 'Month', 'Weekday', 'Daily_Return_%']].head()


In [ ]:
# Rolling averages: 7-day, 30-day, 90-day
df['MA_7'] = df['Close'].rolling(window=7).mean()
df['MA_30'] = df['Close'].rolling(window=30).mean()
df['MA_90'] = df['Close'].rolling(window=90).mean()

df[['Close', 'MA_7', 'MA_30', 'MA_90']].tail()


In [ ]:
# Monthly performance: average return and volume by calendar month
monthly_perf = df.groupby('Month').agg(
    Avg_Daily_Return=('Daily_Return_%', 'mean'),
    Avg_Volume=('Volume', 'mean')
).round(3)
monthly_perf


## 6. Feature Engineering
Price-change features, moving averages, monthly returns, and volatility measures.

In [ ]:
# Daily price change (absolute)
df['Price_Change'] = df['Close'].diff()

# Percentage price change (already computed as Daily_Return_%)

# Monthly returns: % change of month-end close vs previous month-end close
monthly_close = df['Close'].resample('ME').last()
monthly_returns = monthly_close.pct_change() * 100
monthly_returns.name = 'Monthly_Return_%'

# Volatility: rolling 30-day standard deviation of daily returns (annualized)
df['Volatility_30D'] = df['Daily_Return_%'].rolling(window=30).std()
df['Volatility_30D_Annualized'] = df['Volatility_30D'] * np.sqrt(252)

# Cumulative return since the start of the dataset
df['Cumulative_Return_%'] = (df['Close'] / df['Close'].iloc[0] - 1) * 100

df[['Close', 'Price_Change', 'Daily_Return_%', 'Volatility_30D_Annualized', 'Cumulative_Return_%']].tail()


In [ ]:
monthly_returns.tail(12)


## 7. Visualizations

### 7.1 Stock Closing Price Trend

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=1.2)
ax.set_title('AAPL Closing Price Trend (5 Years)')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (USD)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()


### 7.2 Trading Volume Trend

In [ ]:
fig, ax = plt.subplots()
ax.bar(df.index, df['Volume'], color='#ff7f0e', width=1)
ax.set_title('AAPL Trading Volume Trend (5 Years)')
ax.set_xlabel('Date')
ax.set_ylabel('Volume')
plt.tight_layout()
plt.show()


### 7.3 Moving Average Analysis

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df['Close'], label='Close', linewidth=1, alpha=0.6)
ax.plot(df.index, df['MA_7'], label='7-Day MA', linewidth=1.2)
ax.plot(df.index, df['MA_30'], label='30-Day MA', linewidth=1.2)
ax.plot(df.index, df['MA_90'], label='90-Day MA', linewidth=1.2)
ax.set_title('AAPL Closing Price with 7/30/90-Day Moving Averages')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()


### 7.4 Monthly Returns Analysis

In [ ]:
fig, ax = plt.subplots()
colors = ['green' if v >= 0 else 'red' for v in monthly_returns.fillna(0)]
ax.bar(monthly_returns.index, monthly_returns.values, color=colors, width=20)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('AAPL Monthly Returns (%)')
ax.set_xlabel('Month')
ax.set_ylabel('Return (%)')
plt.tight_layout()
plt.show()


### 7.5 Volatility Over Time (Bonus Chart)

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df['Volatility_30D_Annualized'], color='purple', linewidth=1)
ax.set_title('AAPL 30-Day Rolling Annualized Volatility')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility (%)')
plt.tight_layout()
plt.show()


## 8. Key Findings & Conclusions

*(Fill in after running this notebook on your actual downloaded dataset — use the numbers
your charts and tables produce. Suggested prompts to answer:)*

- **Overall trend:** Did AAPL trend up, down, or sideways over the 5-year window? What was the
  total/cumulative return (see `Cumulative_Return_%`)?
- **Volatility:** Which periods showed the highest 30-day annualized volatility? Any visible spikes
  tied to known market events?
- **Moving averages:** Are there clear golden-cross / death-cross points (7-day crossing 30-day or 90-day)?
- **Seasonality:** From `monthly_perf`, which calendar months historically show the strongest/weakest
  average daily returns?
- **Volume:** Do volume spikes line up with large price moves (compare `Volume` against `Daily_Return_%`)?
- **Best/worst months:** Pull the top 3 and bottom 3 rows from `monthly_returns` for headline stats.

**Recommendations:** Summarize 2-3 practical takeaways (e.g., what the moving-average behavior
suggests for a simple trend-following view, or how volatility clustering informs risk framing).
